# M9.9 · Renkli görüntü işleme

Renk biliminden (koniler, CIE ana renkleri, renklilik diyagramı, gamut) renk modellerine: RGB, CMY/CMYK, HSI, HSV, YCbCr, CIE L*a*b*. Elle dönüşümler; saf Python'da HSI ve CMYK gidiş-dönüş; NumPy'da OpenCV'nin 8 bitlik HSV ve YCrCb dönüşümlerini sabit noktalı (fixed-point) tamsayı aritmetiğiyle birebir; PyTorch'ta Di Zenzo vektör gradyanı ve Mahalanobis uzaklığıyla renk bölütleme; en sonda renkleri sıralamadan vektör median filtreyi vektörel yazıyorsun. Ek alıştırmalar sahte renk, renk dilimleme, HSI'da histogram eşitleme ve L*a*b*.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-9-color.html](https://m-gonul.github.io/pytorch-lab/m9-9-color.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_9_color.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"skimage\"}))\n    rng = np.random.default_rng(0)\n    cols = rng.random((300, 3)).tolist() + [[0.3, 0.3, 0.3], [0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0]]\n\n    def fwd():\n        for r, g, b in cols:\n            h, s, i = need(ns, \"rgb_to_hsi\")(r, g, b)\n            close(i, (r + g + b) / 3, what=\"I\")\n            if r + g + b > 0:\n                close(s, 1 - 3 * min(r, g, b) / (r + g + b), atol=1e-12, what=\"S\")\n        close(ns[\"rgb_to_hsi\"](1, 0, 0)[0], 0.0, what=\"kırmızı H\")\n        close(ns[\"rgb_to_hsi\"](0, 1, 0)[0], 120.0, atol=1e-9, what=\"yeşil H\")\n        close(ns[\"rgb_to_hsi\"](0, 0, 1)[0], 240.0, atol=1e-9, what=\"mavi H\")\n        close(ns[\"rgb_to_hsi\"](1, 1, 0)[0], 60.0, atol=1e-9, what=\"sarı H\")\n        ensure(ns[\"rgb_to_hsi\"](0.3, 0.3, 0.3)[0] == 0 and ns[\"rgb_to_hsi\"](0, 0, 0)[1] == 0, \"gri ve siyah özel durumları\")\n    R.test(\"rgb_to_hsi\", fwd)\n\n    def rt():\n        err = 0.0\n        for r, g, b in cols[:300]:\n            back = need(ns, \"hsi_to_rgb\")(*ns[\"rgb_to_hsi\"](r, g, b))\n            err = max(err, max(abs(x - y) for x, y in zip(back, (r, g, b))))\n        ensure(err < 1e-9, f\"gidiş-dönüş hatası {err:.2e}\")\n        return f\"300 rastgele renkte en büyük hata {err:.1e}\"\n    R.test(\"hsi_to_rgb gidiş-dönüş\", rt)\n\n    def cm():\n        close(need(ns, \"rgb_to_cmyk\")(0.2, 0.5, 0.9), (0.8 / 0.9 - 0.1 / 0.9, 0.5 / 0.9 - 0.1 / 0.9, 0.0, 0.1), atol=1e-12, what=\"(0.2, 0.5, 0.9)\")\n        ensure(ns[\"rgb_to_cmyk\"](0, 0, 0) == (0.0, 0.0, 0.0, 1.0), \"siyah: K = 1, diğerleri 0\")\n        err = max(max(abs(x - y) for x, y in zip(need(ns, \"cmyk_to_rgb\")(*ns[\"rgb_to_cmyk\"](r, g, b)), (r, g, b))) for r, g, b in cols)\n        ensure(err < 1e-12, f\"gidiş-dönüş {err:.1e}\")\n    R.test(\"CMYK gidiş-dönüş\", cm)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    rng = np.random.default_rng(0)\n    imgs = {\"astronaut\": data.astronaut(), \"coffee\": data.coffee(), \"rastgele\": rng.integers(0, 256, (128, 128, 3)).astype(np.uint8),\n            \"gri ve uçlar\": np.array([[[0, 0, 0], [255, 255, 255], [128, 128, 128], [255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 0, 255], [1, 0, 0]]], np.uint8)}\n    for name, im in imgs.items():\n        def t(im=im):\n            got = need(ns, \"rgb2hsv8\")(im); want = cv2.cvtColor(im, cv2.COLOR_RGB2HSV)\n            ensure(np.array_equal(got, want), f\"HSV: {(got != want).any(-1).sum()} piksel farklı\")\n            got = need(ns, \"rgb2ycrcb8\")(im); want = cv2.cvtColor(im, cv2.COLOR_RGB2YCrCb)\n            ensure(np.array_equal(got, want), f\"YCrCb: {(got != want).any(-1).sum()} piksel farklı\")\n        R.test(f\"= cv2.cvtColor ({name})\", t)\n\n    def hsi():\n        im = imgs[\"rastgele\"]\n        got = need(ns, \"rgb2hsi\")(im)\n        f = im.astype(np.float64) / 255\n        close(got[..., 2], f.mean(-1), what=\"I\")\n        close(got[..., 1], 1 - 3 * f.min(-1) / f.sum(-1), atol=1e-12, what=\"S\")\n        hsv = cv2.cvtColor(im, cv2.COLOR_RGB2HSV_FULL).astype(np.float64)[..., 0] * 360 / 256\n        d = np.abs((got[..., 0] - hsv + 180) % 360 - 180)\n        ensure(np.median(d) < 6, f\"HSI tonu HSV tonundan çok farklı: medyan {np.median(d):.1f}°\")\n        return f\"HSI tonu ile HSV tonu arasında medyan fark {np.median(d):.1f}° (altıgen ve dairesel ton tanımları farklı)\"\n    R.test(\"rgb2hsi\", hsi)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = torch.tensor(data.astronaut()[100:220, 150:300], dtype=torch.float64).permute(2, 0, 1)[None] / 255\n\n    def eig():\n        Fv, th = need(ns, \"di_zenzo\")(img)\n        np_img = img[0].permute(1, 2, 0).numpy()\n        gx = np.stack([cv2.Sobel(np_img[..., c], cv2.CV_64F, 1, 0, borderType=cv2.BORDER_REPLICATE) for c in range(3)], -1)\n        gy = np.stack([cv2.Sobel(np_img[..., c], cv2.CV_64F, 0, 1, borderType=cv2.BORDER_REPLICATE) for c in range(3)], -1)\n        gxx, gyy, gxy = (gx * gx).sum(-1), (gy * gy).sum(-1), (gx * gy).sum(-1)\n        lmax = 0.5 * (gxx + gyy + np.sqrt((gxx - gyy) ** 2 + 4 * gxy ** 2))\n        close(Fv[0].numpy() ** 2, lmax, atol=1e-9, what=\"F² = en büyük özdeğer\")\n        close(ns[\"channel_sum_grad\"](img)[0].numpy(), np.sqrt(gx ** 2 + gy ** 2).sum(-1), atol=1e-9, what=\"kanal toplamı\")\n    R.test(\"di_zenzo: F² = λ_max\", eig)\n\n    def toy():\n        n = 9\n        step_x = torch.zeros(n, n, dtype=torch.float64); step_x[:, n // 2:] = 1\n        step_y = step_x.T.clone()\n        a = torch.stack([step_x, step_x, step_x])[None]\n        e = torch.stack([step_x, step_x, step_y])[None]\n        c = n // 2\n        sa, se = ns[\"channel_sum_grad\"](a)[0, c, c].item(), ns[\"channel_sum_grad\"](e)[0, c, c].item()\n        fa, fe = need(ns, \"di_zenzo\")(a)[0][0, c, c].item(), ns[\"di_zenzo\"](e)[0][0, c, c].item()\n        ensure(abs(sa - se) < 1e-9, \"kanal toplamları eşit olmalı\")\n        ensure(fa > fe * 1.15, f\"vektör gradyanı (a) {fa:.2f}, (e) {fe:.2f}: (a) daha güçlü olmalı\")\n        R.table(\"Notlardaki örnek: merkez piksel\", [\"görüntü\", \"kanal toplamı\", \"Di Zenzo F\"], [[\"(a) üç kanal aynı kenar\", f\"{sa:.2f}\", f\"{fa:.2f}\"], [\"(e) B kanalı dik kenar\", f\"{se:.2f}\", f\"{fe:.2f}\"]],\n                note=\"Kanal toplamı ikisini ayırt edemiyor; vektör gradyanı (a)'daki renk kenarının daha güçlü olduğunu görüyor.\")\n    R.test(\"kanal toplamı yanılır, vektör gradyanı yanılmaz\", toy)\n\n    def seg():\n        im = torch.tensor(data.astronaut(), dtype=torch.float64).permute(2, 0, 1)[None] / 255\n        samples = im[0, :, 20:60, 360:420].reshape(3, -1).T\n        m = need(ns, \"mahalanobis_mask\")(im, samples, 3.0)\n        a = samples.mean(0).numpy(); C = np.cov(samples.numpy().T)\n        z = im[0].permute(1, 2, 0).numpy() - a\n        d = np.sqrt(np.einsum(\"hwi,ij,hwj->hw\", z, np.linalg.inv(C), z))\n        ensure(np.array_equal(m[0].numpy(), d <= 3.0), \"Mahalanobis maskesi\")\n        b = need(ns, \"box_mask\")(im, samples, 1.25)\n        s = samples.std(0).numpy()\n        ensure(np.array_equal(b[0].numpy(), (np.abs(z) <= 1.25 * s).all(-1)), \"kutu maskesi\")\n        return f\"Mahalanobis (D0 = 3): %{m.double().mean() * 100:.1f}, kutu (1.25σ): %{b.double().mean() * 100:.1f} piksel\"\n    R.test(\"renk bölütleme maskeleri\", seg)\n",
 "e": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import max_loops, need, ensure, bench, fmt_time\n\n\ndef ref_vm(img, k):\n    H, W, _ = img.shape\n    r = k // 2\n    P = np.pad(img.astype(np.float64), ((r, r), (r, r), (0, 0)), mode=\"edge\")\n    out = np.zeros_like(img)\n    for y in range(H):\n        for x in range(W):\n            win = P[y:y + k, x:x + k].reshape(-1, 3)\n            d = np.sqrt(((win[:, None] - win[None]) ** 2).sum(-1)).sum(1)\n            out[y, x] = win[np.argmin(d)]\n    return out\n\n\ndef noisy(seed, H=40, W=48):\n    rng = np.random.default_rng(seed)\n    im = data.astronaut()[200:200 + H, 300:300 + W].copy()\n    m = rng.random((H, W)) < 0.15\n    im[m] = rng.integers(0, 256, (m.sum(), 3))\n    return im\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"en fazla bir döngü (bloklar)\", lambda: max_loops(src, 1, func=\"vector_median\"))\n    for k in (3, 5):\n        def t(k=k):\n            im = noisy(k)\n            got = need(ns, \"vector_median\")(torch.tensor(im), k).numpy()\n            want = ref_vm(im, k)\n            diff = (got != want).any(-1)\n            ensure(diff.mean() < 0.002, f\"{int(diff.sum())} piksel başvurudan farklı\")\n        R.test(f\"vector_median = başvuru ({k} × {k})\", t)\n\n    def novel():\n        im = noisy(7, 64, 64)\n        vm = ns[\"vector_median\"](torch.tensor(im), 3).numpy() if \"vector_median\" in ns else None\n        mm = cv2.medianBlur(im, 3)\n        P = np.pad(im, ((1, 1), (1, 1), (0, 0)), mode=\"edge\")\n        win = np.lib.stride_tricks.sliding_window_view(P, (3, 3), (0, 1)).transpose(0, 1, 3, 4, 2).reshape(64, 64, 9, 3)\n        inwin = lambda out: (win == out[:, :, None, :]).all(-1).any(-1)\n        ensure(vm is not None and inwin(vm).all(), \"vektör median çıkışı penceredeki renklerden biri olmalı\")\n        clean = data.astronaut()[200:264, 300:364].astype(float)\n        e = lambda o: np.abs(o.astype(float) - clean).mean()\n        R.table(\"15% rastgele renk gürültüsü, 3 × 3\", [\"filtre\", \"penceredeki bir renk mi?\", \"ortalama mutlak hata\"],\n                [[\"marjinal median (cv2, kanal kanal)\", f\"%{inwin(mm).mean() * 100:.0f} piksel\", f\"{e(mm):.1f}\"], [\"vektör median\", \"%100 piksel\", f\"{e(vm):.1f}\"]],\n                note=\"Marjinal median kanalları ayrı sıraladığı için pencerede olmayan renkler üretebilir.\")\n    R.test(\"vektör median yeni renk üretmez\", novel)\n    if \"vector_median\" in ns:\n        im = noisy(9, 64, 64)\n        t1 = bench(lambda: ref_vm(im, 3), repeat=1, number=1)\n        t2 = bench(lambda: ns[\"vector_median\"](torch.tensor(im), 3), repeat=3)\n        R.table(\"64 × 64, 3 × 3\", [\"yöntem\", \"süre\"], [[\"piksel döngüsü (NumPy)\", fmt_time(t1)], [\"unfold + cdist\", fmt_time(t2)]])\n",
 "x1": "import numpy as np\nfrom skimage import data\nfrom labcheck import no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()\n\n    def sl():\n        levels = [50, 100, 101, 200]\n        cols = [(i * 50, 255 - i * 50, i * 20) for i in range(5)]\n        got = need(ns, \"intensity_slice\")(img, levels, cols)\n        want = np.zeros(img.shape + (3,), np.uint8)\n        for k in range(5):\n            lo = -1 if k == 0 else levels[k - 1]; hi = 256 if k == 4 else levels[k]\n            want[(img >= max(lo, 0)) & (img < hi)] = cols[k]\n        ensure(got.shape == want.shape and got.dtype == np.uint8 and np.array_equal(got, want), \"dilimleme\")\n        ensure(np.array_equal(ns[\"intensity_slice\"](np.array([[99, 100, 101]], np.uint8), [100], [(0, 0, 0), (9, 9, 9)])[0, :, 0], [0, 9, 9]), \"eşik kendisi üst dilime girer\")\n    R.test(\"intensity_slice\", sl)\n\n    def sp():\n        got = need(ns, \"sin_pseudocolor\")(img, 1.0, (0.0, 2.0, 4.0))\n        f = img.astype(np.float64)\n        for c, p in enumerate((0.0, 2.0, 4.0)):\n            ensure(np.array_equal(got[..., c], np.round(255 * (1 + np.sin(2 * np.pi * f / 255 + p)) / 2).astype(np.uint8)), f\"kanal {c}\")\n    R.test(\"sin_pseudocolor\", sp)\n",
 "x2": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n    img = rng.random((50, 60, 3))\n\n    def cube():\n        a = np.array([0.6, 0.3, 0.2])\n        got = need(ns, \"slice_cube\")(img, a, 0.4)\n        inside = (np.abs(img - a) <= 0.2).all(-1)\n        ensure(np.array_equal(got[inside], img[inside]) and np.all(got[~inside] == 0.5), \"küp\")\n        got = need(ns, \"slice_sphere\")(img, a, 0.3)\n        inside = ((img - a) ** 2).sum(-1) <= 0.09\n        ensure(np.array_equal(got[inside], img[inside]) and np.all(got[~inside] == 0.5), \"küre\")\n        ensure(not np.shares_memory(got, img), \"girdi değiştirilmemeli\")\n    R.test(\"renk dilimleme (küp ve küre)\", cube)\n\n    def eq():\n        im = (data.coffee() * 0.55).astype(np.uint8)\n        a = need(ns, \"equalize_intensity\")(im)\n        b = need(ns, \"equalize_channels\")(im)\n        ensure(a.shape == im.shape and a.dtype == np.uint8, \"uint8, aynı boyut\")\n        h0 = ns[\"rgb2hsi\"](im / 255.0); ha = ns[\"rgb2hsi\"](a / 255.0); hb = ns[\"rgb2hsi\"](b / 255.0)\n        sat = h0[..., 1] > 0.15\n        dh = lambda h: np.median(np.abs((h[..., 0] - h0[..., 0] + 180) % 360 - 180)[sat])\n        ensure(dh(ha) < 3, f\"I eşitlemede ton {dh(ha):.1f}° değişti\")\n        ensure(dh(hb) > dh(ha) + 2, \"kanal kanal eşitleme tonu daha çok bozmalı\")\n        R.table(\"Koyu kahve görüntüsü\", [\"yöntem\", \"ort. I\", \"medyan ton kayması\"], [[\"girdi\", f\"{h0[..., 2].mean():.2f}\", \"—\"], [\"yalnız I eşitle (HSI)\", f\"{ha[..., 2].mean():.2f}\", f\"{dh(ha):.1f}°\"], [\"R, G, B ayrı eşitle\", f\"{hb[..., 2].mean():.2f}\", f\"{dh(hb):.1f}°\"]])\n    R.test(\"HSI'da yoğunluk eşitleme tonu korur\", eq)\n",
 "x3": "import numpy as np\nfrom skimage import color, data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.astronaut()[::4, ::4] / 255.0\n\n    def xyz():\n        close(need(ns, \"rgb_to_xyz\")(img), color.rgb2xyz(img), atol=1e-9, what=\"XYZ\")\n    R.test(\"rgb_to_xyz = skimage.color.rgb2xyz\", xyz)\n\n    def lab():\n        close(need(ns, \"xyz_to_lab\")(ns[\"rgb_to_xyz\"](img)), color.rgb2lab(img), atol=1e-6, what=\"Lab\")\n    R.test(\"Lab = skimage.color.rgb2lab\", lab)\n\n    def chrom():\n        prim = np.array([[[1.0, 0, 0], [0, 1.0, 0], [0, 0, 1.0], [1.0, 1.0, 1.0]]])\n        x, y = need(ns, \"chromaticity\")(ns[\"rgb_to_xyz\"](prim))\n        rows = [[n, f\"{x[0, i]:.3f}\", f\"{y[0, i]:.3f}\"] for i, n in enumerate([\"R\", \"G\", \"B\", \"beyaz (D65)\"])]\n        close(x[0, 3], 0.3127, atol=2e-3, what=\"D65 x\"); close(y[0, 3], 0.3290, atol=2e-3, what=\"D65 y\")\n        R.table(\"sRGB ana renklerinin renklilik koordinatları\", [\"renk\", \"x\", \"y\"], rows, note=\"Bu üç nokta sRGB gamut üçgeninin köşeleri; at nalının tamamını kapsamaz.\")\n    R.test(\"chromaticity\", chrom)\n\n    def de():\n        a = color.rgb2lab(np.array([[[0.5, 0.2, 0.2]]])); b = color.rgb2lab(np.array([[[0.52, 0.2, 0.2]]]))\n        c = color.rgb2lab(np.array([[[0.2, 0.5, 0.2]]])); d = color.rgb2lab(np.array([[[0.2, 0.52, 0.2]]]))\n        close(need(ns, \"delta_e76\")(a, b), color.deltaE_cie76(a, b), what=\"ΔE\")\n        return f\"R'de +0.02: ΔE = {float(ns['delta_e76'](a, b)[0, 0]):.2f}; G'de +0.02: ΔE = {float(ns['delta_e76'](c, d)[0, 0]):.2f} (RGB'de eşit adım, algıda değil)\"\n    R.test(\"delta_e76\", de)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_9_color as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Algı:** retinadaki 6–7 milyon koni üç türdür (≈ %65 kırmızıya, %33 yeşile, %2 maviye duyarlı; mavi koniler en hassası). Bu yüzden renkler **R, G, B** ana renklerinin karışımı olarak görülür. CIE (1931) ana renkleri: mavi 435.8 nm, yeşil 546.1 nm, kırmızı 700 nm. Üç sabit ana renk görünür tüm renkleri üretemez: üçgen **gamut** at nalı biçimli renklilik diyagramını kapsayamaz.

**Nitelikler:** *ton* (baskın dalga boyu), *doygunluk* (beyazla seyrelme; pembe = kırmızı + beyaz), *parlaklık*. Ton + doygunluk = **renklilik**. Trikromatik katsayılar $x=\frac{X}{X+Y+Z}$, $y$, $z$, $x+y+z=1$.

**Toplamsal ve çıkarımsal:** ışıkta R + G + B = beyaz; boyada ikincil renkler C, M, Y ana renktir: $[C,M,Y]=1-[R,G,B]$. Gerçek mürekkepler saf olmadığından siyah ayrı basılır: $K=\min(C,M,Y)$, $C'=\frac{C-K}{1-K}$ (K = 1 ise C′ = M′ = Y′ = 0).

**HSI** (Gonzalez; $[0,1]$ RGB):
$$\theta=\cos^{-1}\frac{\frac12[(R-G)+(R-B)]}{\sqrt{(R-G)^2+(R-B)(G-B)}},\quad H=\begin{cases}\theta&B\le G\\360-\theta&B>G\end{cases}$$
$$S=1-\frac{3\min(R,G,B)}{R+G+B},\qquad I=\frac{R+G+B}{3}.$$
Geri dönüş üç bölgede (RG: $0\le H<120$, GB, BR): örn. RG bölgesinde $B=I(1-S)$, $R=I\big[1+\frac{S\cos H}{\cos(60°-H)}\big]$, $G=3I-(R+B)$; diğerlerinde önce $H$'den 120 ya da 240 çıkarılıp roller döndürülür.

**HSV** (OpenCV): $V=\max$, $S=(V-\min)/V$, $H$ altıgen üzerinde parçalı doğrusal; 8 bitte $H\in[0,180)$. **YCbCr:** parlaklık + iki renk farkı (JPEG, video). **L*a*b*:** cihazdan bağımsız, algısal olarak yaklaşık düzgün; yoğunluğu renkten ayırır.

**Tam renkli işleme:** her kanalı ayrı işlemek ile vektörü işlemek **doğrusal** komşuluk işlemlerinde aynıdır (ortalama, Laplacian); median, kenar ve eşitleme gibi doğrusal olmayanlarda değil. **Renk bölütleme:** örnek renklerin ortalaması $a$; $D(z,a)=\sqrt{(z-a)^TC^{-1}(z-a)}\le D_0$ (Mahalanobis; $C=I$ ise küre).

**Di Zenzo gradyanı:** $g_{xx}=|\partial_x\mathbf c|^2$, $g_{yy}=|\partial_y\mathbf c|^2$, $g_{xy}=\partial_x\mathbf c\cdot\partial_y\mathbf c$; yön $\theta=\frac12\tan^{-1}\frac{2g_{xy}}{g_{xx}-g_{yy}}$, büyüklük
$$F_\theta=\sqrt{\tfrac12[(g_{xx}+g_{yy})+(g_{xx}-g_{yy})\cos2\theta+2g_{xy}\sin2\theta]}.$$

:::tanım Üç renk katsayıları ve metamerizm
Bir rengi üç birincil rengin (CIE X, Y, Z tristimulus değerleri) karışımı olarak yazarız; **trikromatik katsayılar** bunların oranlarıdır: $x=\frac{X}{X+Y+Z}$, $y=\frac{Y}{X+Y+Z}$, $z=1-x-y$. $(x,y)$ ikilisi rengin parlaklıktan bağımsız "rengini" (renklilik, chromaticity) verir: CIE renklilik diyagramı bu düzlemdir; bir cihazın gösterebildiği renkler (gamut) bu diyagramda birincil renklerinin oluşturduğu üçgendir.

**Metamerizm:** spektral olarak **farklı** iki ışığın aynı X, Y, Z değerlerini (dolayısıyla aynı algılanan rengi) üretmesi. Göz üç tür koniyle sonsuz boyutlu bir spektrumu üç sayıya indirdiği için bilgi kaybeder; bu kayıp hem renkli ekranların (üç birincil renkle her rengi taklit etmek) hem de "dükkânda aynı, güneşte farklı" kumaş sorununun (aydınlatma değişince metamerler ayrışır) nedenidir.
:::

## 1 · Elle

<!-- çözüm -->

HSI'da $\theta$: pay $\frac12(0.4+0.6)=0.5$, payda $\sqrt{0.16+0.6\cdot0.2}=0.5292$, $\theta=\cos^{-1}0.9449=19.1°$; $B\le G$ olduğu için $H=\theta$.

Küp dilimlemede $|0.42-0.30|=0.12>W/2$: piksel ilgi bölgesinin dışında, her kanal nötr 0.5'e gider.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**CMY ve CMYK ($[0,1]$): RGB = (0.2, 0.5, 0.9)**

- `c`: C
- `k`: K = min(C, M, Y)
- `cp`: C′ = (C − K)/(1 − K)

**HSI: RGB = (0.8, 0.4, 0.2)**

- `h`: H (derece)
- `s`: S
- `i`: I

**Diğer**

- `y`: RGB (200, 100, 50) için Y = 0.299R + 0.587G + 0.114B
- `cr`: Cr = 0.713(R − Y) + 128
- `d`: Öklid uzaklığı: (200, 100, 50) ile a = (180, 110, 40)
- `sl`: Küp dilimleme, a = (0.6, 0.3, 0.2), W = 0.2: (0.65, 0.42, 0.18) pikselinin G çıkışı (dışarıdaysa nötr 0.5)

In [ ]:
cevaplar = dict(
    c=None,
    k=None,
    cp=None,
    h=None,
    s=None,
    i=None,
    y=None,
    cr=None,
    d=None,
    sl=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"c":[0.8,0],"k":[0.1,0],"cp":[0.777778,0.0001],"h":[19.1066,0.001],"s":[0.571429,0.0001],"i":[0.466667,0.0001],"y":[124.2,0.001],"cr":[182.0454,0.001],"d":[24.4949,0.001],"sl":[0.5,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, tek piksel fonksiyonları, değerler $[0,1]$:

- `rgb_to_hsi(r, g, b)` → `(H, S, I)`, $H$ derece $[0,360)$. Gri piksel ($R=G=B$) için $H=0$, siyah için $S=0$.
- `hsi_to_rgb(h, s, i)` — üç bölgeli formül.
- `rgb_to_cmyk(r, g, b)` ve `cmyk_to_rgb(c, m, y, k)`.

Kontrol, rastgele renklerde gidiş-dönüş hatasını ölçer.

In [ ]:
%%writefile stage_b.py
import math


def rgb_to_hsi(r, g, b):
    ...


def hsi_to_rgb(h, s, i):
    ...


def rgb_to_cmyk(r, g, b):
    ...


def cmyk_to_rgb(c, m, y, k):
    ...


print(rgb_to_hsi(0.8, 0.4, 0.2))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**, `uint8` RGB → `uint8`, OpenCV ile **birebir**.

**`rgb2hsv8(img)`** (`color_hsv`, 12 bitlik sabit nokta):

- `v = max`, `vmin = min`, `diff = v − vmin`.
- Tablolar: `sdiv[i] = round((255 << 12) / i)`, `hdiv[i] = round((180 << 12) / (6 i))` ($i>0$; $i=0$ için 0).
- `s = (diff · sdiv[v] + 2048) >> 12`.
- `h = g − b` (v = r ise), değilse `b − r + 2·diff` (v = g ise), değilse `r − g + 4·diff`; `h = (h · hdiv[diff] + 2048) >> 12`; negatifse `+180`.

**`rgb2ycrcb8(img)`** (14 bitlik sabit nokta): `Y = (4899 R + 9617 G + 1868 B + 8192) >> 14`; `δ = (128 << 14) + 8192`; `Cr = ((R − Y)·11682 + δ) >> 14`; `Cb = ((B − Y)·9241 + δ) >> 14`; 0–255'e kırp. (11682 = 0.713·2¹⁴, 9241 = 0.564·2¹⁴.)

**`rgb2hsi(img)`** — b'nin vektörel hali, `float64` `(H, W, 3)`: H derece, S, I.

`>>` NumPy'da negatif sayılarda da aritmetik kaydırmadır (C'deki gibi), `int64` kullan.

In [ ]:
%%writefile stage_c.py
import numpy as np


def rgb2hsv8(img):
    ...


def rgb2ycrcb8(img):
    ...


def rgb2hsi(img):
    ...


from skimage import data
print(rgb2hsv8(data.astronaut())[0, 0])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

`img (B, 3, H, W)` float.

- `di_zenzo(img)` → `(F, theta)`: her kanalda Sobel (gruplu `conv2d`, `groups=3`, `replicate` dolgu), $g_{xx}, g_{yy}, g_{xy}$ kanallar üzerinden toplam; $\theta=\frac12$`atan2`$(2g_{xy}, g_{xx}-g_{yy})$; $F_\theta$ formülü. `F` ve `theta` `(B, H, W)`.
- `channel_sum_grad(img)` — kanalların Sobel büyüklüklerinin toplamı (karşılaştırma için).
- `mahalanobis_mask(img, samples, D0)` — `samples (N, 3)`; $a$ ortalama, $C$ kovaryans (`torch.cov`, yansız), $\sqrt{(z-a)^TC^{-1}(z-a)}\le D_0$ → `(B, H, W)` bool.
- `box_mask(img, samples, k=1.25)` — her kanalda $|z_j-a_j|\le k\sigma_j$ (yansız std).

Kontrol, $F_\theta^2$'nin $2\times2$ matrisin en büyük özdeğerine eşit olduğunu ve notlardaki örneği (aynı kanal gradyanları, farklı renk kenarı) sınar.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def di_zenzo(img):
    ...


def channel_sum_grad(img):
    ...


def mahalanobis_mask(img, samples, D0):
    ...


def box_mask(img, samples, k=1.25):
    ...


x = torch.rand(1, 3, 16, 16)
print(di_zenzo(x)[0].shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Vektör median:** penceredeki renk vektörleri $\{c_i\}$ arasından $\sum_j\lVert c_i-c_j\rVert_2$'yi en küçük yapan $c_i$ (eşitlikte ilk). Çıkış her zaman penceredeki renklerden biridir.

`vector_median(img, k)` — `img (H, W, 3)` uint8 tensör → aynı tip. Kenar `replicate`. `F.unfold` → `(H·W, k², 3)`, `torch.cdist` → `(H·W, k², k²)`, satır toplamı, `argmin`, `gather`. Bellek $HW\cdot k^4$: büyük görüntüde satır bloklarına bölmek serbest (tek bir döngü).

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def vector_median(img, k):
    """img (H, W, 3) uint8 tensör."""
    ...


print(vector_median(torch.randint(0, 256, (8, 8, 3), dtype=torch.uint8), 3).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Sahte renk: yoğunluk dilimleme ve sinüs dönüşümleri

- `intensity_slice(img, levels, colors)` — `levels` artan eşikler $l_1<…<l_P$, `colors` $P+1$ RGB üçlüsü: $f<l_1$ → `colors[0]`, $l_k\le f<l_{k+1}$ → `colors[k]`, $f\ge l_P$ → `colors[P]`. `(H, W, 3)` uint8.
- `sin_pseudocolor(img, freq, phases)` — her kanal $c=\mathrm{round}\big(255\cdot\frac{1+\sin(2\pi\,\text{freq}\,f/255+\varphi_c)}2\big)$; Gonzalez'in "üç bağımsız dönüşüm" yaklaşımı.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def intensity_slice(img, levels, colors):
    ...


def sin_pseudocolor(img, freq, phases):
    ...


from skimage import data
print(intensity_slice(data.camera(), [64, 128, 192], [(0, 0, 0), (255, 0, 0), (0, 255, 0), (255, 255, 255)]).shape)

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Renk dilimleme ve HSI'da histogram eşitleme

Hazır `rgb2hsi` ve `hsi2rgb` verildi.

- `slice_cube(img, a, W)` — $[0,1]$ float; her kanalda $|r_j-a_j|>W/2$ olan pikseller 0.5 (nötr gri), diğerleri aynen.
- `slice_sphere(img, a, R0)` — $\sum_j(r_j-a_j)^2>R_0^2$ ise 0.5.
- `equalize_intensity(img)` — `uint8` RGB → HSI; $I$'yı 256 seviyede eşitle (M9.2 formülü: $\frac{L-1}{N}\mathrm{cdf}$ yuvarlanmış, sonra /255), H ve S'yi koru, geri çevir, `uint8`.
- `equalize_channels(img)` — her kanala `cv2.equalizeHist` (karşılaştırma için).

In [ ]:
%%writefile extra_x2.py
import numpy as np
import cv2


def rgb2hsi(f):
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    tot = r + g + b
    S = np.where(tot > 0, 1 - 3 * f.min(-1) / np.where(tot > 0, tot, 1), 0.0)
    den = np.sqrt((r - g) ** 2 + (r - b) * (g - b))
    th = np.degrees(np.arccos(np.clip(0.5 * ((r - g) + (r - b)) / np.where(den > 0, den, 1), -1, 1)))
    return np.stack([np.where(den > 0, np.where(b <= g, th, 360 - th), 0.0), S, tot / 3], -1)


def hsi2rgb(hsi):
    H, S, I = hsi[..., 0] % 360, hsi[..., 1], hsi[..., 2]
    out = np.zeros(hsi.shape)
    for lo, order in ((0, (2, 0, 1)), (120, (0, 1, 2)), (240, (1, 2, 0))):
        m = (H >= lo) & (H < lo + 120)
        h = np.radians(H - lo)
        x = I * (1 - S)
        y = I * (1 + S * np.cos(h) / np.cos(np.radians(60) - h))
        z = 3 * I - (x + y)
        for ch, v in zip(order, (x, y, z)):
            out[..., ch] = np.where(m, v, out[..., ch])
    return out


def slice_cube(img, a, W):
    ...


def slice_sphere(img, a, R0):
    ...


def equalize_intensity(img):
    ...


def equalize_channels(img):
    ...


from skimage import data
print(equalize_intensity(data.coffee()).shape)

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · CIE XYZ, renklilik ve L*a*b* (ΔE)

sRGB (D65) → L*a*b*, `skimage.color.rgb2lab` ile aynı:

- `srgb_to_linear(c)` — $c\le0.04045$ ise $c/12.92$, değilse $((c+0.055)/1.055)^{2.4}$.
- `rgb_to_xyz(img)` — doğrusal RGB × `[[0.412453, 0.357580, 0.180423], [0.212671, 0.715160, 0.072169], [0.019334, 0.119193, 0.950227]]`.
- `xyz_to_lab(xyz)` — beyaz $(0.95047, 1, 1.08883)$; $f(q)=q^{1/3}$ ($q>0.008856$), değilse $7.787q+16/116$; $L=116f(Y)-16$, $a=500(f(X)-f(Y))$, $b=200(f(Y)-f(Z))$.
- `chromaticity(xyz)` → $(x, y)$.
- `delta_e76(lab1, lab2)` — Öklid uzaklığı.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def srgb_to_linear(c):
    ...


def rgb_to_xyz(img):
    ...


def xyz_to_lab(xyz):
    ...


def chromaticity(xyz):
    ...


def delta_e76(lab1, lab2):
    ...


print(xyz_to_lab(rgb_to_xyz(np.array([[[1.0, 1.0, 1.0]]]))))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Renkli görüntü renklendirme (colorization)

[Zhang, Isola, Efros — Colorful Image Colorization (ECCV 2016)](https://arxiv.org/abs/1603.08511)

L*a*b* uzayında L kanalından ab kanallarını tahmin eden bir CNN eğit. ab düzlemini 313 bölmeye nicemle, regresyon yerine çok sınıflı sınıflandırma ve nadir renkleri yeniden ağırlıklandırma (class rebalancing) uygula; tahmini dağılımdan "annealed mean" ile renk seç.


**Adımlar**

1. Bu modüldeki L*a*b* dönüşümünü kullanarak veri hattı (ör. bir ImageNet alt kümesi ya da Places).
2. ab nicemleme ızgarası (10'luk adım, gamut içi bölmeler) ve yumuşak kodlama (5 komşu, Gauss ağırlık).
3. Ampirik renk dağılımından sınıf ağırlıkları $w\propto((1-\lambda)\tilde p+\lambda/Q)^{-1}$.
4. Regresyon (L2) taban çizgisiyle karşılaştır: L2'nin neden soluk, kahverengimsi sonuç verdiğini göster.

**Kabul ölçütleri**

- Aynı ağda L2 regresyon ve sınıflandırma çıktılarının görsel karşılaştırması.
- Ortalama doygunluk (HSV S) tablosu: sınıflandırma > regresyon.
- Annealed mean sıcaklığının (T = 1, 0.38, 0) etkisi gösterilmiş.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 6](https://www.imageprocessingplace.com/) — ©
- [OpenCV — Color conversions (imgproc/src/color_hsv, color_yuv)](https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html) — Apache-2.0
- [Di Zenzo — A note on the gradient of a multi-image (CVGIP 1986)](https://doi.org/10.1016/0734-189X(86)90223-9) — ©
- [Astola, Haavisto, Neuvo — Vector median filters (Proc. IEEE 1990)](https://doi.org/10.1109/5.54807) — ©
- [scikit-image — color](https://scikit-image.org/docs/stable/api/skimage.color.html) — BSD-3